<a href="https://colab.research.google.com/github/SsemuliJoseph/Sunbird/blob/main/sunflower_14B_4bit_quantized_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sunbird/Sunflower-14B-4-bit Quantized

## Usage
### Installation

In [ ]:
!pip install torch transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 20.2 MB/s eta 0:00:00


### Usage

In [ ]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os
from getpass import getpass

In [ ]:
os.environ["HF_TOKEN"] = getpass("HF_TOKEN: ")

HF_TOKEN: ··········


In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("Sunbird/Sunflower-14B-4bit-nf4-bnb")

# Configure quantization (already quantized, but need config for loading)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", #fp4
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    "Sunbird/Sunflower-14B-4bit-nf4-bnb",
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/5.40k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/219 [00:00<?, ?B/s]

In [ ]:
SYSTEM_MESSAGE = """You are Sunflower, a multilingual assistant for Ugandan languages made by Sunbird AI. You specialise in accurate translations, explanations, summaries and other cross-lingual tasks."""
prompt_text = "Sunbird AI is a non-profit research organization in Kampala, Uganda. We build and deploy practical applied machine learning systems for African contexts."
user_prompt = f"Translate to Luganda: {prompt_text}"
messages = [
    {"role": "system", "content": SYSTEM_MESSAGE},
    {"role": "user", "content": user_prompt}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
text_streamer = transformers.TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

print(f'\n\n{user_prompt}\n---')

# Increasing num_beams may increase quality of output, but reduces speed.
num_beams = 1
generation_config = {
    "max_new_tokens": 512,
    "temperature": 0.5,
    "do_sample": True,
    "no_repeat_ngram_size": 5,
    "num_beams": num_beams,
}


outputs = model.generate(
  **inputs,
  **generation_config,
  streamer = text_streamer if num_beams==1 else None,
)

if num_beams > 1:
    response = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
    print(response)




Translate to Luganda: Sunbird AI is a non-profit research organization in Kampala, Uganda. We build and deploy practical applied machine learning systems for African contexts.
---
Sunbird AI kibiina ekikola okunoonyereza mu Kampala, Uganda, nga kigenderera okukola n’okussa mu nkola enkola z’ebyuma ezikozesebwa mu mbeera z’abantu ku lukalu lwa Afirika.
